In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.marchenko.lesson2 import Exercise

In [ ]:
def normalize_data(X_train, X_val, X_test):
    """Стандартизация признаков на основе статистик обучающей выборки."""
    # Вычисляем статистики только на тренировочных данных
    feature_means = X_train.mean(axis=0)
    feature_stds = X_train.std(axis=0)

    # Защита от константных признаков
    feature_stds[feature_stds == 0] = 1.0

    # Нормализация всех выборок
    X_train_norm = (X_train - feature_means) / feature_stds
    X_val_norm = (X_val - feature_means) / feature_stds
    X_test_norm = (X_test - feature_means) / feature_stds

    return X_train_norm, X_val_norm, X_test_norm

In [ ]:
# Загружаем данные
iris = load_iris()
X, y = iris.data, (iris.target == 0).astype(int)

print(f"Данные: {X.shape}, Setosa: {sum(y == 1)}, Другие: {sum(y == 0)}")
print(f"Признаки: {iris.feature_names}")

# Разделяем 60-20-20
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=100, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=100, stratify=y_temp)

print(
    f"\nРазмеры: Train {len(X_train)} ({len(X_train) / len(X) * 100:.0f}%), "
    f"Val {len(X_val)} ({len(X_val) / len(X) * 100:.0f}%), "
    f"Test {len(X_test)} ({len(X_test) / len(X) * 100:.0f}%)"
)

# Нормализация
X_train_norm, X_val_norm, X_test_norm = normalize_data(X_train, X_val, X_test)

In [ ]:
def evaluate_config(learning_rate, batch_size, X_train, X_val, y_train, y_val, n_ep=25, random_seed=100):
    """Обучает модель логистической регрессии и оценивает на валидации."""
    # Создаем модель
    rng = np.random.default_rng(random_seed)
    model = Exercise.create_logistic_model(num_features=X_train.shape[1], rng=rng)

    # Обучаем
    Exercise.fit(model, X_train, y_train, lr=learning_rate, n_ep=n_ep, batch_size=batch_size)

    # Метрики
    accuracy = model.metric(X_val, y_val, metric_type="ACCURACY")
    auroc = model.metric(X_val, y_val, metric_type="AUROC")

    return accuracy, auroc, model

In [ ]:
# Подбор гиперпараметров
n_ep = 25
seeds = [100, 200, 300, 400]
lr_values = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0]
batch_sizes = [1, 2, 4, 8, 16, 32, 64, None]

results = []

for lr in lr_values:
    for bs in batch_sizes:
        auroc_scores, acc_scores = [], []

        for seed in seeds:
            acc, auroc, _ = evaluate_config(lr, bs, X_train_norm, X_val_norm, y_train, y_val, n_ep, seed)
            auroc_scores.append(auroc)
            acc_scores.append(acc)

        results.append(
            {
                "lr": lr,
                "bs": bs,
                "auroc_mean": np.mean(auroc_scores),
                "auroc_std": np.std(auroc_scores),
                "acc_mean": np.mean(acc_scores),
                "acc_std": np.std(acc_scores),
            }
        )

# Сортируем по AUROC
results.sort(key=lambda x: (-x["auroc_mean"], x["auroc_std"]))

# Вывод топ-5
print("\n" + "=" * 70)
print("ТОП-5 ПО AUROC")
print("=" * 70)
print(f"{'lr':<8} {'batch':<8} {'AUROC':<12} {'ACC':<12}")
print("-" * 70)
for r in results[:5]:
    bs = "None" if r["bs"] is None else r["bs"]
    print(f"{r['lr']:<8.3f} {bs:<8} {r['auroc_mean']:.4f}±{r['auroc_std']:.4f}  {r['acc_mean']:.4f}±{r['acc_std']:.4f}")

print("\n" + "=" * 70)
print("ТОП-5 ПО ACCURACY")
print("=" * 70)
print(f"{'lr':<8} {'batch':<8} {'ACC':<12} {'AUROC':<12}")
print("-" * 70)
for r in sorted(results, key=lambda x: -x["acc_mean"])[:5]:
    bs = "None" if r["bs"] is None else r["bs"]
    print(f"{r['lr']:<8.3f} {bs:<8} {r['acc_mean']:.4f}±{r['acc_std']:.4f}  {r['auroc_mean']:.4f}±{r['auroc_std']:.4f}")

# Лучшая конфигурация
best = max(results, key=lambda x: x["auroc_mean"])
print("\n" + "=" * 70)
print("ЛУЧШАЯ КОНФИГУРАЦИЯ")
print("=" * 70)
print(f"Learning Rate:  {best['lr']}")
print(f"Batch Size:     {'None' if best['bs'] is None else best['bs']}")
print(f"Val AUROC:      {best['auroc_mean']:.4f} ± {best['auroc_std']:.4f}")
print(f"Val Accuracy:   {best['acc_mean']:.4f} ± {best['acc_std']:.4f}")

# Финальный тест
final_rng = np.random.default_rng(42)
final_model = Exercise.create_logistic_model(num_features=X_train_norm.shape[1], rng=final_rng)
Exercise.fit(final_model, X_train_norm, y_train, lr=best["lr"], n_ep=n_ep, batch_size=best["bs"])

test_auroc = final_model.metric(X_test_norm, y_test, metric_type="AUROC")
test_acc = final_model.metric(X_test_norm, y_test, metric_type="ACCURACY")

print(f"\nTest AUROC:     {test_auroc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Gap:            {best['auroc_mean'] - test_auroc:.4f}")